# 仕様

最終課題用のノートブックのなかに、犬と猫の画像を学習したモデルを作成して、分類を行なうプログラムを作成してください。

- 本レッスン内容で学習した流れに沿って、深層学習プログラムを作成してください
- データの前処理や水増しの処理を入れてください
- MobileNetV2 のモデルを利用してください（画像サイズは MobileNetV2 が対応する大きさへのリサイズが必要です）
- 必ず最後に evaluate() を実行して、正答率がわかるようにしてください。

In [24]:
import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(96, 96),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(96, 96),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

class_names = train_dataset.class_names


def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

train_dataset = train_dataset.shuffle(32)


input_layer = tf.keras.Input(shape=(96, 96, 3))
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(96, 96, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False

output_layer = tf.keras.layers.Dense(1, activation='sigmoid')

model = tf.keras.Sequential([
    base_model,
    output_layer
])

model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])

model.fit(train_dataset, epochs=5)

Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.7961 - loss: 0.4200
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - accuracy: 0.9289 - loss: 0.1813
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.9550 - loss: 0.1346
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 60ms/step - accuracy: 0.9672 - loss: 0.1067
Epoch 5/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.9750 - loss: 0.0879


In [25]:
print(model.evaluate(test_dataset))

# pred_data = model.predict(test_dataset)
# pred_data

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.9900 - loss: 0.0427
[0.042696528136730194, 0.9900000095367432]
